## CS496 Code-Switching — Experiment Runner
Run all cells top to bottom. Only change values in the **Config** cell.

In [ ]:
# ============================================================
#  CONFIG — only edit this cell
# ============================================================

# Model name — pick one:
#   GitHub Models : "gpt-4o"
#   Anthropic     : "claude-sonnet-4-20250514"
#   Google        : "gemini-2.5-flash"
#   Groq          : "llama-3.3-70b-versatile"
MODEL_NAME = "gpt-4o"

# Your API key for the model above
API_KEY = "sk-..."

# Base URL — set for your provider:
#   GitHub Models : "https://models.inference.ai.azure.com"
#   Groq          : "https://api.groq.com/openai/v1"
#   Anthropic     : ""   (leave empty)
#   Google        : ""   (leave empty)
BASE_URL = "https://models.inference.ai.azure.com"

# Experiment type: "lid" or "mt"
EXPERIMENT = "lid"

# Prompting strategy: "zero_shot" or "few_shot"
STRATEGY = "zero_shot"

# Sentences per API call (batch size)
BATCH_SIZE = 5

# Paths (do not change)
SAMPLE_PATH   = "experiments/data/sample_500.json"
PROMPT_PATH   = f"experiments/prompts/{STRATEGY}_{EXPERIMENT}.txt"
RESULTS_DIR   = "experiments/results"
MODEL_SLUG    = MODEL_NAME.replace("/", "-").replace(":", "-")
RAW_PATH      = f"{RESULTS_DIR}/{MODEL_SLUG}_{STRATEGY}_{EXPERIMENT}_raw.json"
PARSED_PATH   = f"{RESULTS_DIR}/{MODEL_SLUG}_{STRATEGY}_{EXPERIMENT}_parsed.json"

print(f"Model     : {MODEL_NAME}")
print(f"Provider  : {BASE_URL or 'native SDK'}")
print(f"Experiment: {EXPERIMENT}")
print(f"Strategy  : {STRATEGY}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Raw out   : {RAW_PATH}")
print(f"Parsed out: {PARSED_PATH}")

## 1. Install & Imports

In [ ]:
!pip install openai anthropic google-generativeai groq -q

import os, json, time, re
from datetime import datetime


## 2. Load Data & Prompt

In [ ]:
# Load 500-sentence sample
with open(SAMPLE_PATH, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

records = sample_data["records"]
print(f"Loaded {len(records)} records")
print(f"Dataset breakdown: {sample_data['dataset_breakdown']}")

# Load prompt template
with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    PROMPT_TEMPLATE = f.read()

print(f"\nPrompt template loaded ({len(PROMPT_TEMPLATE)} chars)")
print("\nFirst 200 chars of prompt:")
print(PROMPT_TEMPLATE[:200])

# Create results directory
os.makedirs(RESULTS_DIR, exist_ok=True)


## 3. API Caller (Model-Agnostic)

In [ ]:
def detect_provider(model_name: str, base_url: str) -> str:
    """Detect which API provider to use based on model name and base_url."""
    m = model_name.lower()
    if "claude" in m:
        return "anthropic"
    elif "gemini" in m:
        return "google"
    elif base_url:
        return "openai_compatible"  # covers GitHub Models, Groq, and any OpenAI-compatible API
    else:
        raise ValueError(f"Cannot detect provider for model: {model_name}")

PROVIDER = detect_provider(MODEL_NAME, BASE_URL)
print(f"Detected provider: {PROVIDER}")

# Initialize client
if PROVIDER == "openai_compatible":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

elif PROVIDER == "anthropic":
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)

elif PROVIDER == "google":
    import google.generativeai as genai
    genai.configure(api_key=API_KEY)
    client = genai.GenerativeModel(MODEL_NAME)

print(f"Client initialized for {PROVIDER}")

In [ ]:
def call_api(prompt: str, max_retries: int = 5) -> str:
    """
    Call the API with exponential backoff retry logic.
    Returns the raw response text.
    """
    for attempt in range(max_retries):
        try:
            if PROVIDER == "openai_compatible":
                response = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0,
                    max_tokens=2048,
                )
                return response.choices[0].message.content

            elif PROVIDER == "anthropic":
                response = client.messages.create(
                    model=MODEL_NAME,
                    max_tokens=2048,
                    messages=[{"role": "user", "content": prompt}],
                )
                return response.content[0].text

            elif PROVIDER == "google":
                response = client.generate_content(
                    prompt,
                    generation_config={"temperature": 0.0, "max_output_tokens": 2048}
                )
                return response.text

        except Exception as e:
            wait = 2 ** attempt
            print(f"  [Attempt {attempt+1}/{max_retries}] Error: {e}. Retrying in {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"API call failed after {max_retries} attempts.")

print("call_api() defined.")

## 4. Prompt Builder

In [ ]:
def build_prompt(batch: list) -> str:
    """
    Build the prompt for a batch of records.
    Replaces the placeholder at the end of the template with actual sentences.
    """
    sentences_block = "\n".join(
        f'sent_{i+1}: "{r["text"]}"'
        for i, r in enumerate(batch)
    )
    # Replace the placeholder line at the bottom of the template
    prompt = PROMPT_TEMPLATE.replace(
        'sent_{sent_num}: "{sentence}"',
        sentences_block
    )
    return prompt

# Test it
test_prompt = build_prompt(records[:2])
print("Sample prompt (last 300 chars):")
print(test_prompt[-300:])


## 5. Response Parser

In [ ]:
VALID_LABELS = {"AR", "EN", "AR-LAT", "OTHER", "OL"}

def parse_response(raw: str, batch: list, experiment: str) -> dict:
    """
    Parse raw API response into structured output.
    Returns a dict with parsed results and a parse_status field.
    """
    result = {
        "parse_status": "ok",
        "parse_error": None,
        "sentences": []
    }

    # Strip markdown code fences if present
    cleaned = raw.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"```\s*$", "", cleaned)
    cleaned = cleaned.strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        result["parse_status"] = "json_error"
        result["parse_error"] = str(e)
        return result

    if not isinstance(parsed, list):
        result["parse_status"] = "format_error"
        result["parse_error"] = "Response is not a JSON array"
        return result

    for i, item in enumerate(parsed):
        if i >= len(batch):
            break
        record = batch[i]

        if experiment == "lid":
            tokens_out = item.get("tokens", [])

            # Check token count match
            if len(tokens_out) != record["n_tokens"]:
                result["parse_status"] = "token_count_mismatch"
                result["parse_error"] = (
                    f"sent_{i+1}: expected {record['n_tokens']} tokens, "
                    f"got {len(tokens_out)}"
                )

            # Check for invalid labels
            for t in tokens_out:
                if t.get("label") not in VALID_LABELS:
                    result["parse_status"] = "invalid_label"
                    result["parse_error"] = f"Invalid label: {t.get('label')}"

            result["sentences"].append({
                "id": record["id"],
                "dataset": record["dataset"],
                "text": record["text"],
                "predicted_tokens": tokens_out,
                "gold_labels": record["gold_labels"],
                "n_tokens_expected": record["n_tokens"],
                "n_tokens_predicted": len(tokens_out),
            })

        elif experiment == "mt":
            translation = item.get("translation", "")
            result["sentences"].append({
                "id": record["id"],
                "dataset": record["dataset"],
                "text": record["text"],
                "translation": translation,
            })

    return result

print("parse_response() defined.")


## 6. Test Run (5 Sentences)
Run this first before the full run. Check the output carefully.

In [ ]:
TEST_RECORDS = records[:5]
test_batch_prompt = build_prompt(TEST_RECORDS)

print("Calling API with 5 test sentences...")
test_raw = call_api(test_batch_prompt)

print("\nRaw response:")
print(test_raw)

print("\nParsed output:")
test_parsed = parse_response(test_raw, TEST_RECORDS, EXPERIMENT)
print(f"Parse status: {test_parsed['parse_status']}")
if test_parsed["parse_error"]:
    print(f"Parse error: {test_parsed['parse_error']}")
for s in test_parsed["sentences"]:
    print(f"\n{s['id']}: {s['text'][:60]}...")
    if EXPERIMENT == "lid":
        print(f"  Predicted: {[t['label'] for t in s['predicted_tokens']]}")
        print(f"  Gold     : {s['gold_labels']}")
    elif EXPERIMENT == "mt":
        print(f"  Translation: {s['translation']}")


## 7. Full Run (500 Sentences)
Only run this after the test run looks correct.

In [ ]:
# Split into batches
batches = [records[i:i+BATCH_SIZE] for i in range(0, len(records), BATCH_SIZE)]
print(f"Total batches: {len(batches)} (batch size: {BATCH_SIZE})")
print(f"Total sentences: {len(records)}")

all_raw = []
all_parsed = []
parse_failures = 0
run_start = datetime.now().isoformat()

for batch_idx, batch in enumerate(batches):
    print(f"Batch {batch_idx+1}/{len(batches)}...", end=" ")

    prompt = build_prompt(batch)

    try:
        raw_text = call_api(prompt)
    except RuntimeError as e:
        print(f"FAILED: {e}")
        for r in batch:
            all_raw.append({
                "batch_idx": batch_idx,
                "sentence_ids": [r["id"] for r in batch],
                "raw_response": None,
                "error": str(e),
                "timestamp": datetime.now().isoformat()
            })
        parse_failures += len(batch)
        continue

    # Log raw response
    all_raw.append({
        "batch_idx": batch_idx,
        "sentence_ids": [r["id"] for r in batch],
        "raw_response": raw_text,
        "error": None,
        "timestamp": datetime.now().isoformat()
    })

    # Parse response
    parsed = parse_response(raw_text, batch, EXPERIMENT)

    if parsed["parse_status"] != "ok":
        print(f"PARSE ISSUE ({parsed['parse_status']}): {parsed['parse_error']}")
        parse_failures += len(batch)
    else:
        print("ok")

    all_parsed.extend(parsed["sentences"])

    # Small delay to avoid rate limits
    time.sleep(0.5)

print(f"\nDone. Parse failures: {parse_failures}/{len(records)}")
print(f"Parse failure rate: {parse_failures/len(records)*100:.1f}%")


## 8. Save Results

In [ ]:
# Save raw responses
raw_output = {
    "model": MODEL_NAME,
    "provider": PROVIDER,
    "experiment": EXPERIMENT,
    "strategy": STRATEGY,
    "batch_size": BATCH_SIZE,
    "run_start": run_start,
    "run_end": datetime.now().isoformat(),
    "total_sentences": len(records),
    "parse_failures": parse_failures,
    "parse_failure_rate": round(parse_failures / len(records) * 100, 2),
    "batches": all_raw
}

with open(RAW_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_output, f, ensure_ascii=False, indent=2)
print(f"Raw responses saved to: {RAW_PATH}")

# Save parsed outputs
parsed_output = {
    "model": MODEL_NAME,
    "provider": PROVIDER,
    "experiment": EXPERIMENT,
    "strategy": STRATEGY,
    "total_sentences": len(records),
    "parse_failures": parse_failures,
    "parse_failure_rate": round(parse_failures / len(records) * 100, 2),
    "sentences": all_parsed
}

with open(PARSED_PATH, "w", encoding="utf-8") as f:
    json.dump(parsed_output, f, ensure_ascii=False, indent=2)
print(f"Parsed outputs saved to: {PARSED_PATH}")


## 9. Push Results to GitHub

In [ ]:
GITHUB_TOKEN = "your_token_here"
GITHUB_USER  = "abojoudah"
GITHUB_REPO  = "cs496-codeswitching"

!git add experiments/results/
!git commit -m "Add {MODEL_SLUG}_{STRATEGY}_{EXPERIMENT} results"
!git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git main
